
# 🧩 02 - Funções de cadastro (CRUD básico) - MySQL

Este notebook contém as funções principais para trabalhar com a tabela `pessoas`:

- Cadastrar nova pessoa
- Listar todas as pessoas
- Buscar pessoas pelo nome
- Atualizar telefone/contato
- Alterar status (ex.: inativar / encaminhar)

Ele utiliza:

- O arquivo de configuração `mysql_config.json` criado no `00_mysql_config.ipynb`
- As tabelas criadas no `01_config_db.ipynb`


In [1]:

import mysql.connector
from datetime import datetime

%run "./00_mysql_config.ipynb"

def get_connection():
    cfg = carregar_config_mysql()
    return mysql.connector.connect(
        host=cfg["host"],
        port=cfg["port"],
        user=cfg["user"],
        password=cfg["password"],
        database=cfg["database"]
    )

def cadastrar_pessoa(
    nome: str,
    apelido: str = None,
    data_nascimento: str = None,
    documento_principal: str = None,
    tem_documentos: bool = False,
    telefone: str = None,
    contato_emergencia: str = None,
    cidade_origem: str = None,
    situacao_rua_desde: str = None,
    saude_resumo: str = None,
    dependencias_quimicas: str = None,
    observacoes: str = None,
    status: str = "ativo"
) -> int:
    conn = get_connection()
    cur = conn.cursor()

    insert_sql = '''
    INSERT INTO pessoas (
        nome,
        apelido,
        data_nascimento,
        documento_principal,
        tem_documentos,
        telefone,
        contato_emergencia,
        cidade_origem,
        situacao_rua_desde,
        saude_resumo,
        dependencias_quimicas,
        observacoes,
        status,
        data_cadastro
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    '''

    tem_docs_int = 1 if tem_documentos else 0
    data_nasc = data_nascimento or None

    valores = (
        nome,
        apelido,
        data_nasc,
        documento_principal,
        tem_docs_int,
        telefone,
        contato_emergencia,
        cidade_origem,
        situacao_rua_desde,
        saude_resumo,
        dependencias_quimicas,
        observacoes,
        status,
        datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    )

    cur.execute(insert_sql, valores)
    conn.commit()
    pessoa_id = cur.lastrowid

    cur.close()
    conn.close()

    print(f"[OK] Pessoa cadastrada com ID: {pessoa_id}")
    return pessoa_id

def listar_pessoas(apenas_ativas: bool = True):
    conn = get_connection()
    cur = conn.cursor(dictionary=True)

    if apenas_ativas:
        cur.execute("SELECT * FROM pessoas WHERE status = 'ativo' ORDER BY id")
    else:
        cur.execute("SELECT * FROM pessoas ORDER BY id")

    rows = cur.fetchall()
    cur.close()
    conn.close()

    print(f"[INFO] {len(rows)} registro(s) encontrado(s).")
    for r in rows:
        print(r)

    return rows

def buscar_por_nome(parte_nome: str):
    conn = get_connection()
    cur = conn.cursor(dictionary=True)

    sql = "SELECT * FROM pessoas WHERE LOWER(nome) LIKE %s ORDER BY id"
    cur.execute(sql, (f"%{parte_nome.lower()}%",))

    rows = cur.fetchall()
    cur.close()
    conn.close()

    print(f"[INFO] {len(rows)} registro(s) encontrado(s).")
    for r in rows:
        print(r)

    return rows

def atualizar_contato(pessoa_id: int, telefone: str = None, contato_emergencia: str = None):
    conn = get_connection()
    cur = conn.cursor()

    campos = []
    valores = []

    if telefone is not None:
        campos.append("telefone = %s")
        valores.append(telefone)

    if contato_emergencia is not None:
        campos.append("contato_emergencia = %s")
        valores.append(contato_emergencia)

    if not campos:
        print("[WARN] Nenhum campo de contato informado para atualização.")
        cur.close()
        conn.close()
        return

    valores.append(pessoa_id)
    sql = f"UPDATE pessoas SET {', '.join(campos)} WHERE id = %s"

    cur.execute(sql, valores)
    conn.commit()
    cur.close()
    conn.close()

    print(f"[OK] Contatos atualizados para o ID: {pessoa_id}")

def atualizar_status(pessoa_id: int, novo_status: str):
    conn = get_connection()
    cur = conn.cursor()

    sql = "UPDATE pessoas SET status = %s WHERE id = %s"
    cur.execute(sql, (novo_status, pessoa_id))

    conn.commit()
    cur.close()
    conn.close()

    print(f"[OK] Status atualizado para '{novo_status}' no ID: {pessoa_id}")


[OK] Configuração salva em: D:\Acx - Dev\Projeto Social\mysql_config.json


c:\Users\André Camargo\AppData\Local\Programs\Python\Python311\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)
